In [2]:
"""
Production-ready example of TOOLS in LangChain.

Run inside the langchainupdates (Python 3.12) venv, with GROQ_API_KEY in .env.

What this demonstrates:
  1. Defining tools cleanly (docstrings + typed args = the model's instructions).
  2. Tools that can fail, with errors handled gracefully instead of crashing.
  3. Letting `create_agent` run the invoke -> run tools -> invoke loop for you.
  4. Inspecting which tools the model actually called (observability).
"""

import os
import logging
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain.agents import create_agent

# ---------------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------------
load_dotenv()  # loads GROQ_API_KEY from .env

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger("tools_example")


# ---------------------------------------------------------------------------
# 1. Define tools
#    The docstring and type hints are what the MODEL reads. Be precise:
#    a clear docstring is the difference between the tool being called at
#    the right time with the right args, and being misused.
# ---------------------------------------------------------------------------

@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together. Use this for any arithmetic addition
    instead of computing it yourself, to guarantee correctness."""
    return a + b


class WeatherInput(BaseModel):
    """Schema for the get_weather tool — each field's description guides the model."""
    city: str = Field(description="City name, e.g. 'Kathmandu' or 'Pokhara'")
    units: str = Field(
        default="celsius",
        description="Temperature units: 'celsius' or 'fahrenheit'",
    )


@tool(args_schema=WeatherInput)
def get_weather(city: str, units: str = "celsius") -> str:
    """Get the current weather for a city. Call this whenever the user asks
    about current temperature, conditions, or weather in a specific place."""
    # --- In production this hits a real weather API. Stubbed for the example. ---
    # A real version would: validate the city, call the API, handle timeouts,
    # and return a clean string. We simulate a possible failure below.
    fake_db = {"kathmandu": 22, "pokhara": 25, "lalitpur": 23}
    key = city.strip().lower()
    if key not in fake_db:
        # Returning a clear message (not raising) lets the model recover and
        # tell the user, rather than the whole run crashing.
        return f"No weather data available for '{city}'."
    temp_c = fake_db[key]
    if units == "fahrenheit":
        return f"It's {round(temp_c * 9 / 5 + 32)}°F and clear in {city}."
    return f"It's {temp_c}°C and clear in {city}."


@tool
def get_word_length(word: str) -> int:
    """Return the number of letters in a single word. Use this when asked to
    count characters or letters, instead of guessing."""
    return len(word)


# ---------------------------------------------------------------------------
# 2. Build the agent
#    `create_agent` runs the full loop for you:
#      model -> (tool calls?) -> execute tools -> model -> final answer,
#    repeating as many times as the model needs.
# ---------------------------------------------------------------------------

def build_agent():
    if not os.getenv("GROQ_API_KEY"):
        raise RuntimeError(
            "GROQ_API_KEY not found. Add it to your .env file "
            "(get one free at https://console.groq.com)."
        )

    tools = [add_numbers, get_weather, get_word_length]

    agent = create_agent(
        model="groq:qwen/qwen3-32b",
        tools=tools,
        system_prompt=(
            "You are a helpful assistant. Use the provided tools when they are "
            "relevant. Do not invent weather data or do arithmetic yourself — "
            "always use the tools for those."
        ),
    )
    return agent


# ---------------------------------------------------------------------------
# 3. Run it and inspect what happened
# ---------------------------------------------------------------------------

def ask(agent, question: str) -> str:
    """Send one question through the agent and return the final text answer.
    Also logs which tools the model decided to call (useful in production)."""
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    messages = result["messages"]

    # Observability: log every tool the model invoked and the result it got.
    for m in messages:
        tool_calls = getattr(m, "tool_calls", None)
        if tool_calls:
            for call in tool_calls:
                log.info("Tool requested: %s args=%s", call["name"], call["args"])
        if m.__class__.__name__ == "ToolMessage":
            log.info("Tool result:    %s", m.content)

    # The final assistant message is the answer.
    return messages[-1].content


def main():
    agent = build_agent()

    questions = [
        "What's the weather in Pokhara in fahrenheit?",
        "How many letters are in the word 'parrot'?",
        "What is 137 plus 489?",
        "What's the weather in Tokyo?",   # not in our stub -> graceful 'no data'
    ]

    for q in questions:
        print("\n" + "=" * 70)
        print(f"USER: {q}")
        try:
            answer = ask(agent, q)
            print(f"ASSISTANT: {answer}")
        except Exception as e:
            # Top-level guard so one bad question doesn't kill the whole run.
            log.error("Failed to answer %r: %s", q, e)


if __name__ == "__main__":
    main()


USER: What's the weather in Pokhara in fahrenheit?


INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO | Tool requested: get_weather args={'city': 'Pokhara', 'units': 'fahrenheit'}
INFO | Tool result:    It's 77°F and clear in Pokhara.


ASSISTANT: The current weather in Pokhara is **77°F** with clear skies. Let me know if you'd like additional details! ☀️

USER: How many letters are in the word 'parrot'?


INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO | Tool requested: get_word_length args={'word': 'parrot'}
INFO | Tool result:    6


ASSISTANT: The word "parrot" has **6 letters**.

USER: What is 137 plus 489?


INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO | Tool requested: add_numbers args={'a': 137, 'b': 489}
INFO | Tool result:    626.0


ASSISTANT: The sum of 137 and 489 is **626**.

USER: What's the weather in Tokyo?


INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO | Tool requested: get_weather args={'city': 'Tokyo', 'units': 'celsius'}
INFO | Tool result:    No weather data available for 'Tokyo'.


ASSISTANT: It seems that weather data for Tokyo isn't available at the moment. This could be due to a temporary service issue or limitations in the data source. You might want to check a dedicated weather service or app for the most accurate and up-to-date information. Let me know if you'd like help with anything else!


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
response=model.invoke("Why do parrot talk?")
response

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots. Parrots are known for their ability to mimic human speech, right? But why do they do that? I need to break this down.\n\nFirst, maybe it\'s related to their natural behavior. In the wild, parrots live in social groups. They probably use vocalizations to communicate with each other. So when they\'re in captivity, talking might be a way to interact with humans as their primary social group. That makes sense because they might imitate humans as a form of social bonding.\n\nThen there\'s the aspect of learning. Parrots are highly intelligent birds. They have strong memory and can learn through observation. They might mimic sounds to get attention or to be rewarded, like getting treats or affection. Positive reinforcement could be a big part of their learning process.\n\nAnother angle is that talking helps them adapt. In the wild, they mimic other birds or environme

In [2]:
from langchain.tools import tool


@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])


In [6]:
response=model_with_tools.invoke("what's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")


INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. Boston is the location here. So I\'ll call the function with "Boston" as the location. Make sure the JSON is correctly formatted with the name and arguments.\n', 'tool_calls': [{'id': '4b0pq6cja', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 154, 'total_tokens': 251, 'completion_time': 0.14681351, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.006251977, 'prompt_tokens_details': None, 'queue_time': 0.054168743, 'total_time': 0.153065487}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_p

### Tool Execution Loop


In [ ]:
#step1:Model generates tool calls
messages=[{"role":"user","content":"What's the weather in Boston?"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

#step2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)
#step3:Pass results back to model for final response
final_response=model_with_tools.invoke(messages)
print(final_response.text)

The current weather in Boston is sunny. The temperature isn't included in this response, but the conditions are clear! ☀️ Let me know if you'd like to check again or need additional details.


In [5]:
messages

[{'role': 'user', 'content': "What's the weather and temperature in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather and temperature in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user mentioned Boston, I need to call this function with "Boston" as the location. I should make sure the parameters are correctly formatted as per the tool\'s requirements. The required parameter is location, and it\'s a string. So the tool call should be straightforward. I don\'t see any other functions provided, so I\'ll proceed with get_weather.\n', 'tool_calls': [{'id': 'e6nwc095d', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 155, 'total_tokens': 286, 'completion_time': 0.202007592, 'completion_tokens_details': {'reasoning_tokens'